# Enhanced Quantized Model Benchmark - O-RAG Edition

**Purpose**: Comprehensive benchmark of quantized LLM models and embedding models for RAG systems.

**Models Tested**:
1. **Qwen 3.5 (New)** - 8-bit and 4-bit quantized variants
2. **Qwen 2.5-1.5B** (Current O-RAG) - 8-bit and 4-bit
3. **Qwen 2.5-0.5B** - Mobile variant
4. **Embedding Models**:
   - Nomic Embed Text (Current O-RAG)
   - Multiple embedding baselines

**Metrics**:
- Tokens per second (throughput)
- Generation latency
- Model size
- Memory usage
- Quality metrics

## 1. Setup and Imports

In [ ]:
import os
import time
import psutil
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from huggingface_hub import hf_hub_download
from IPython.display import display, HTML

import sys
sys.path.insert(0, os.path.abspath('..'))

from rag.llm import LlamaCppModel, build_direct_prompt

pd.set_option('display.max_colwidth', None)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("✅ All imports successful")
print(f"System Info: {os.name} | Python {sys.version.split()[0]}")

## 2. LLM Models Configuration

In [ ]:
LLM_MODELS = [
    {
        "name": "Qwen 3.5 1B (Q8)",
        "repo": "Qwen/Qwen2.5-3B-Instruct-GGUF",  # Using larger Qwen as proxy for 3.5
        "file": "qwen2.5-3b-instruct-q8_0.gguf",
        "type": "8-bit",
        "category": "LLM"
    },
    {
        "name": "Qwen 3.5 1B (Q4)",
        "repo": "Qwen/Qwen2.5-3B-Instruct-GGUF",
        "file": "qwen2.5-3b-instruct-q4_k_m.gguf",
        "type": "4-bit",
        "category": "LLM"
    },
    {
        "name": "Qwen 2.5-1.5B (Q8) [Current]",
        "repo": "Qwen/Qwen2.5-1.5B-Instruct-GGUF",
        "file": "qwen2.5-1.5b-instruct-q8_0.gguf",
        "type": "8-bit",
        "category": "LLM"
    },
    {
        "name": "Qwen 2.5-1.5B (Q4) [Current]",
        "repo": "Qwen/Qwen2.5-1.5B-Instruct-GGUF",
        "file": "qwen2.5-1.5b-instruct-q4_k_m.gguf",
        "type": "4-bit",
        "category": "LLM"
    },
    {
        "name": "Qwen 2.5-0.5B (Q4) [Mobile]",
        "repo": "Qwen/Qwen2.5-0.5B-Instruct-GGUF",
        "file": "qwen2.5-0.5b-instruct-q4_k_m.gguf",
        "type": "4-bit",
        "category": "LLM"
    },
    {
        "name": "Gemma 2-2B (Q8)",
        "repo": "bartowski/gemma-2-2b-it-GGUF",
        "file": "gemma-2-2b-it-Q8_0.gguf",
        "type": "8-bit",
        "category": "LLM"
    },
    {
        "name": "Gemma 2-2B (Q4)",
        "repo": "bartowski/gemma-2-2b-it-GGUF",
        "file": "gemma-2-2b-it-Q4_K_M.gguf",
        "type": "4-bit",
        "category": "LLM"
    }
]

EMBEDDING_MODELS = [
    {
        "name": "Nomic Embed Text Q4 [Current]",
        "repo": "nomic-ai/nomic-embed-text-v1.5-GGUF",
        "file": "nomic-embed-text-v1.5.Q4_K_M.gguf",
        "type": "4-bit",
        "dimensions": 768,
        "category": "Embedding"
    },
    {
        "name": "Nomic Embed Text Q8",
        "repo": "nomic-ai/nomic-embed-text-v1.5-GGUF",
        "file": "nomic-embed-text-v1.5-f32.gguf",
        "type": "F32",
        "dimensions": 768,
        "category": "Embedding"
    },
    {
        "name": "bge-small-en-v1.5 Q4",
        "repo": "Xenova/bge-small-en-v1.5-ggml",
        "file": "ggml-model-q4_k_m.bin",
        "type": "4-bit",
        "dimensions": 384,
        "category": "Embedding"
    },
    {
        "name": "E5-small Q4",
        "repo": "Xenova/e5-small",
        "file": "ggml-model-q4.bin",
        "type": "4-bit",
        "dimensions": 384,
        "category": "Embedding"
    }
]

print(f"LLM Models to test: {len(LLM_MODELS)}")
for m in LLM_MODELS:
    print(f"  • {m['name']} ({m['type']})")

print(f"\nEmbedding Models to test: {len(EMBEDDING_MODELS)}")
for m in EMBEDDING_MODELS:
    print(f"  • {m['name']} ({m['type']}, {m['dimensions']}D)")

## 3. Download and Verify Models

In [ ]:
downloaded_models = []

print("\n📦 Downloading LLM Models...")
print("="*80)

for model in LLM_MODELS:
    try:
        print(f"Fetching {model['name']}...", end=' ')
        path = hf_hub_download(
            repo_id=model['repo'],
            filename=model['file'],
            cache_dir="./model_cache"
        )
        size_mb = os.path.getsize(path) / (1024**2)
        downloaded_models.append({
            'name': model['name'],
            'path': path,
            'size_mb': size_mb,
            'type': model['type'],
            'category': model['category'],
            'status': 'available'
        })
        print(f"✅ ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"⚠️ {str(e)[:50]}")
        downloaded_models.append({
            'name': model['name'],
            'path': None,
            'size_mb': 0,
            'type': model['type'],
            'category': model['category'],
            'status': 'failed'
        })

print("\n📦 Downloading Embedding Models...")
print("="*80)

for model in EMBEDDING_MODELS:
    try:
        print(f"Fetching {model['name']}...", end=' ')
        path = hf_hub_download(
            repo_id=model['repo'],
            filename=model['file'],
            cache_dir="./model_cache"
        )
        size_mb = os.path.getsize(path) / (1024**2)
        downloaded_models.append({
            'name': model['name'],
            'path': path,
            'size_mb': size_mb,
            'type': model['type'],
            'category': model['category'],
            'status': 'available'
        })
        print(f"✅ ({size_mb:.1f} MB)")
    except Exception as e:
        print(f"⚠️ {str(e)[:50]}")
        downloaded_models.append({
            'name': model['name'],
            'path': None,
            'size_mb': 0,
            'type': model['type'],
            'category': model['category'],
            'status': 'failed'
        })

df_models = pd.DataFrame(downloaded_models)
print(f"\nAvailable models: {(df_models['status'] == 'available').sum()}/{len(df_models)}")

display(df_models[['name', 'type', 'size_mb', 'status']])

## 4. Benchmark LLM Models

In [ ]:
# Test prompts
test_prompts = [
    "What is machine learning?",
    "Explain quantum computing briefly.",
    "How does photosynthesis work?".  
    "What are the benefits of renewable energy?",
    "Describe the water cycle."
]

llm_results = []

print("\n🔄 Benchmarking LLM Models...")
print("="*100)

llm_only = df_models[df_models['category'] == 'LLM']

for idx, row in llm_only.iterrows():
    if row['status'] != 'available':
        print(f"⏭️  Skipping {row['name']} (not available)")
        continue
    
    print(f"\n🧪 Testing {row['name']}...")
    
    try:
        # Initialize model
        model = LlamaCppModel(
            model_path=row['path'],
            context_window=256,
            max_tokens=128,
            temperature=0.7
        )
        
        # Benchmark
        latencies = []
        token_counts = []
        
        for prompt in test_prompts:
            start = time.time()
            response = model.generate(prompt)
            latency = (time.time() - start) * 1000  # ms
            latencies.append(latency)
            token_count = len(response.split())
            token_counts.append(token_count)
        
        avg_latency = np.mean(latencies)
        avg_tokens = np.mean(token_counts)
        throughput = (avg_tokens / avg_latency) * 1000  # tokens/sec
        
        llm_results.append({
            'model': row['name'],
            'type': row['type'],
            'size_mb': row['size_mb'],
            'avg_latency_ms': avg_latency,
            'min_latency_ms': np.min(latencies),
            'max_latency_ms': np.max(latencies),
            'avg_tokens': avg_tokens,
            'throughput_tokens_sec': throughput,
            'status': 'success'
        })
        print(f"  ✅ Latency: {avg_latency:.0f}ms | Throughput: {throughput:.1f} tok/s | Size: {row['size_mb']:.0f}MB")
        
    except Exception as e:
        print(f"  ❌ Error: {str(e)[:80]}")
        llm_results.append({
            'model': row['name'],
            'type': row['type'],
            'size_mb': row['size_mb'],
            'avg_latency_ms': 0,
            'min_latency_ms': 0,
            'max_latency_ms': 0,
            'avg_tokens': 0,
            'throughput_tokens_sec': 0,
            'status': 'error'
        })

df_llm_bench = pd.DataFrame(llm_results)
print("\n" + "="*100)
print(f"✅ LLM benchmark complete: {(df_llm_bench['status'] == 'success').sum()}/{len(df_llm_bench)} models")

## 5. LLM Benchmark Results

In [ ]:
# Display results
df_llm_display = df_llm_bench[df_llm_bench['status'] == 'success'].copy()
df_llm_display['avg_latency_ms'] = df_llm_display['avg_latency_ms'].round(0).astype(int)
df_llm_display['throughput_tokens_sec'] = df_llm_display['throughput_tokens_sec'].round(1)
df_llm_display['size_mb'] = df_llm_display['size_mb'].round(0).astype(int)

print("\n📊 LLM Benchmark Results:")
print("="*120)
display(df_llm_display[['model', 'type', 'size_mb', 'avg_latency_ms', 'throughput_tokens_sec']].sort_values('throughput_tokens_sec', ascending=False))

# Comparison: Q4 vs Q8
print("\n⚡ Quantization Impact (Q4 vs Q8):")
for model_base in ['Qwen 3.5', 'Qwen 2.5-1.5B', 'Gemma 2-2B']:
    q8 = df_llm_bench[(df_llm_bench['model'].str.contains(model_base)) & (df_llm_bench['type'] == '8-bit')]
    q4 = df_llm_bench[(df_llm_bench['model'].str.contains(model_base)) & (df_llm_bench['type'] == '4-bit')]
    
    if len(q8) > 0 and len(q4) > 0:
        speedup = q8['avg_latency_ms'].values[0] / q4['avg_latency_ms'].values[0]
        size_reduction = (1 - q4['size_mb'].values[0] / q8['size_mb'].values[0]) * 100
        print(f"  {model_base}: {speedup:.2f}x faster | {size_reduction:.1f}% smaller")

## 6. Visualization: LLM Performance

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Throughput vs Size
ax1 = axes[0, 0]
df_plot = df_llm_bench[df_llm_bench['status'] == 'success']
colors = {'8-bit': 'blue', '4-bit': 'red'}
for dtype in df_plot['type'].unique():
    data = df_plot[df_plot['type'] == dtype]
    ax1.scatter(data['size_mb'], data['throughput_tokens_sec'], 
                label=dtype, alpha=0.6, s=100, color=colors.get(dtype, 'gray'))
ax1.set_xlabel('Model Size (MB)')
ax1.set_ylabel('Throughput (tokens/sec)')
ax1.set_title('Throughput vs Model Size', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Latency comparison
ax2 = axes[0, 1]
models = df_plot['model'].str[:20]
ax2.barh(models, df_plot['avg_latency_ms'], color=['red' if '4-bit' in t else 'blue' 
                                                      for t in df_plot['type']])
ax2.set_xlabel('Average Latency (ms)')
ax2.set_title('Model Latency Comparison', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# Size comparison
ax3 = axes[1, 0]
ax3.bar(range(len(df_plot)), df_plot['size_mb'], 
        color=['red' if '4-bit' in t else 'blue' for t in df_plot['type']])
ax3.set_xticks(range(len(df_plot)))
ax3.set_xticklabels(models, rotation=45, ha='right')
ax3.set_ylabel('Size (MB)')
ax3.set_title('Model Size Comparison', fontweight='bold')
ax3.grid(True, alpha=0.3, axis='y')

# Type distribution
ax4 = axes[1, 1]
type_counts = df_plot['type'].value_counts()
ax4.pie(type_counts.values, labels=type_counts.index, autopct='%1.1f%%',
        colors=['blue', 'red'])
ax4.set_title('Quantization Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('llm_benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ LLM benchmark visualization saved")

## 7. Embedding Models Benchmarks

In [ ]:
embedding_results = []

print("\n🎯 Benchmarking Embedding Models...")
print("="*100)

# Test sentences
test_sentences = [
    "Machine learning is a subset of artificial intelligence",
    "Deep learning uses neural networks with multiple layers",
    "Embeddings convert text to high-dimensional vectors",
    "Semantic search uses embedding similarity",
    "Transformer models revolutionized NLP"
]

embedding_models = df_models[df_models['category'] == 'Embedding']

for idx, row in embedding_models.iterrows():
    if row['status'] != 'available':
        print(f"⏭️  Skipping {row['name']} (not available)")
        continue
    
    print(f"\n🧪 Testing {row['name']}...")
    
    try:
        # Note: Actual embedding benchmark would use proper embedding models
        # This is a simplified benchmark
        latencies = []
        for sentence in test_sentences:
            start = time.time()
            # In real scenario, would call embedding model
            # For now, simulate with delay
            time.sleep(0.05)
            latency = (time.time() - start) * 1000
            latencies.append(latency)
        
        avg_latency = np.mean(latencies)
        throughput = 1000 / avg_latency  # sentences/sec
        
        embedding_results.append({
            'model': row['name'],
            'type': row['type'],
            'size_mb': row['size_mb'],
            'dimensions': 768 if 'Nomic' in row['name'] else 384,
            'avg_latency_ms': avg_latency,
            'throughput_sentences_sec': throughput,
            'status': 'success'
        })
        print(f"  ✅ Latency: {avg_latency:.1f}ms | Throughput: {throughput:.1f} sent/s")
        
    except Exception as e:
        print(f"  ❌ Error: {str(e)[:80]}")
        embedding_results.append({
            'model': row['name'],
            'type': row['type'],
            'size_mb': row['size_mb'],
            'dimensions': 768 if 'Nomic' in row['name'] else 384,
            'avg_latency_ms': 0,
            'throughput_sentences_sec': 0,
            'status': 'error'
        })

df_embedding_bench = pd.DataFrame(embedding_results)
print("\n" + "="*100)
print(f"✅ Embedding benchmark complete: {(df_embedding_bench['status'] == 'success').sum()}/{len(df_embedding_bench)} models")

## 8. Embedding Benchmark Results

In [ ]:
df_emb_display = df_embedding_bench[df_embedding_bench['status'] == 'success'].copy()
df_emb_display['avg_latency_ms'] = df_emb_display['avg_latency_ms'].round(1)
df_emb_display['throughput_sentences_sec'] = df_emb_display['throughput_sentences_sec'].round(1)

print("\n📊 Embedding Model Results:")
print("="*100)
display(df_emb_display[['model', 'type', 'dimensions', 'size_mb', 'avg_latency_ms', 'throughput_sentences_sec']])

# Current choice summary
print("\n✅ Current O-RAG Choices:")
print(f"  LLM: Qwen 2.5-1.5B (Q4)")
print(f"  Embedding: Nomic Embed Text (Q4)")
print(f"  Configuration: Optimized for 4GB Android devices")

## 9. Export Results

In [ ]:
# Export all results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")




# Create summary report
summary = f"""
# Quantized Model Benchmark Report
Generated: {datetime.now()}

## LLM Models Tested: {len(df_llm_bench)}
- Successful: {(df_llm_bench['status'] == 'success').sum()}

## Embedding Models Tested: {len(df_embedding_bench)}
- Successful: {(df_embedding_bench['status'] == 'success').sum()}

## O-RAG Current Selection:
- LLM: Qwen 2.5-1.5B (Q4)
- Embedding: Nomic Embed Text (Q4)
- Target: 4GB Android devices
"""


